In [1]:
from vcm_model import  *

In [2]:
##评估指标说明：
##每个指标对应的list按照如下顺序输入：
##第一个元素为指标类型，type1:原子指标；type2:策略不影响分母；type3:策略影响分母
##第二个元素为指标分子（原子指标即为指标本身）
##第三个元素为分子的协变量，list形式，无协变量则置空
##第四个元素为指标分母
##第五个元素为分母的协变量，list形式，无协变量则置空
eval_dict={
    '冒泡数':['type1','bubble_cnt',[]],
    '发单数':['type1','compete_call_cnt_hxz',[]],
    '完单数':['type1','hxz_finish_cnt',['bubble_cnt','valid_tsh']],
    '应答率':['type3','hxz_answer_cnt',['bubble_cnt','valid_tsh'],'compete_call_cnt_hxz',['bubble_cnt','valid_tsh']],
    '成交率':['type3','hxz_finish_cnt',['bubble_cnt','valid_tsh'],'compete_call_cnt_hxz',['bubble_cnt','valid_tsh']],
    '应答后司机取消率(含改派)':['type3','hxz_grab_after_dri_cancel_cnt',['bubble_cnt','valid_tsh'],'hxz_answer_cnt',['bubble_cnt','valid_tsh']],
    '应答后司机取消率(不含改派)':['type3','hxz_grab_after_dri_cancel_cnt_without_reassign',['bubble_cnt','valid_tsh'],'hxz_answer_cnt',['bubble_cnt','valid_tsh']],
    '应答后乘客取消率':['type3','hxz_grab_after_pas_cancel_cnt',['bubble_cnt','valid_tsh'],'hxz_answer_cnt',['bubble_cnt','valid_tsh']],
    '自营完单数':['type1','hxz_zy_finish_cnt',['bubble_cnt','valid_tsh']],
    '宏鹄完单数':['type1','hxz_hh_finish_cnt',['bubble_cnt','valid_tsh']],
    'API完单数':['type1','hxz_api_finish_cnt',['bubble_cnt','valid_tsh']]
}

In [3]:
##评估指标说明：
##每个指标对应的list按照如下顺序输入：
##第一个元素为指标类型，type1:原子指标；type2:策略不影响分母；type3:策略影响分母
##第二个元素为指标分子（原子指标即为指标本身）
##第三个元素为分子的协变量，list形式，无协变量则置空
##第四个元素为指标分母
##第五个元素为分母的协变量，list形式，无协变量则置空
# eval_dict={
#     'GMV':['type1','gmv',['发单量','TSH'],],
#     '单均GMV':['type2','gmv',['TSH'],'发单量'],
#     '应答后取消率':['type3','应答后取消量',[],'应答量',['发单量','TSH']],
# }

In [4]:
##数据格式说明：
##city_id：城市id，必须项
##tdate：实验日期，必须项
##exp_group：实验分组情况，枚举值treatment_group、control_group，必须项
##minf：时间片标识，必须项
##以上必须项的字段名称和实验分组名称，需与demo数据保持一致
df = pd.read_csv('data2.csv')
def chenge_group_name(x):
    return x + '_group'
df['exp_group'] = df['ab_group'].apply(chenge_group_name)
df.head()

,city_id,tdate,minf,ab_group,hxz_finish_cnt,hxz_answer_cnt,hxz_answer_cnt_without_reassign,hxz_grab_after_dri_cancel_cnt,hxz_grab_after_dri_cancel_cnt_without_reassign,hxz_grab_after_pas_cancel_cnt,hxz_grab_after_pas_cancel_cnt_without_reassign,hxz_gmv_amt,hxz_zy_answer_cnt,hxz_zy_finish_cnt,hxz_hh_answer_cnt,hxz_hh_finish_cnt,hxz_api_finish_cnt,pbd_answer_cnt,pbd_finish_cnt,pbd_zy_answer_cnt,pbd_zy_finish_cnt,pbd_hh_answer_cnt,pbd_hh_finish_cnt,pbd_api_finish_cnt,thlm_answer_cnt,thlm_finish_cnt,thlm_zy_answer_cnt,thlm_zy_finish_cnt,thlm_hh_answer_cnt,thlm_hh_finish_cnt,thlm_api_finish_cnt,juhe_zyhh_finish_cnt,juhe_zy_finish_cnt,juhe_zy_gmv_amt,compete_call_cnt_unity,compete_call_cnt_hxz,compete_call_cnt_pbd,compete_call_cnt_zy,compete_call_cnt_hh,compete_call_cnt_zyhh,compete_call_cnt_sf,compete_call_cnt_hxz_zy,compete_call_cnt_hxz_hh,compete_call_cnt_hxz_sf,compete_call_cnt_pbd_zy,compete_call_cnt_pbd_hh,compete_call_cnt_pbd_sf,bubble_cnt,valid_tsh,exp_group
0,36,2025-03-18,0,treatment,594,747,707,57,16,95,87,14249.25,147,101,32,20,419,359,279,86,62,2,1,12,610,487,152,113,24,14,360,311,276,5450.52,6386,772,2669,5785,3370,5813,5265,758,753,767,2468,272,1701,1173,194.683333,treatment_group
1,36,2025-03-18,2,treatment,410,544,506,58,15,75,71,9167.77,91,50,29,17,275,262,200,52,37,0,0,6,444,314,91,62,24,19,233,185,149,3028.90,4631,566,1945,4250,2573,4269,3787,559,558,565,1791,255,1203,758,111.783333,treatment_group
2,36,2025-03-18,4,treatment,216,267,262,12,4,39,38,4750.78,31,18,7,6,153,163,116,32,18,1,0,3,199,144,42,28,10,5,111,75,64,1803.22,2525,313,1054,2369,1462,2383,2089,312,311,312,988,154,670,418,64.616667,treatment_group
3,36,2025-03-18,7,control,854,1101,1044,76,11,167,162,19464.59,203,123,65,47,536,490,364,119,76,11,8,25,1036,717,219,141,81,18,558,413,340,7178.45,13604,1374,3883,12396,9445,12464,11328,1369,1358,1367,3467,1086,1972,1911,220.700000,control_group
4,36,2025-03-18,11,control,921,1183,1119,94,12,163,154,19732.42,234,139,26,18,700,457,316,108,77,4,2,24,734,554,170,118,18,11,425,365,334,6018.78,10282,1306,3744,9137,5911,9205,8013,1279,1262,1293,3349,555,1766,1889,293.383333,control_group


In [5]:
##评估结果将自动输出到该目录下的Result文件夹中
import warnings
warnings.filterwarnings("ignore")
vcm = vcm_result(data=df,evaluate_dict=eval_dict)
vcm.run()

  0%|          | 0/9 [00:00<?, ?it/s]

36



  0%|          | 0/11 [00:00<?, ?it/s]

冒泡数



  9%|▉         | 1/11 [00:00<00:06,  1.44it/s]

发单数


... (208 outputs left out)


 91%|█████████ | 10/11 [00:26<00:02,  2.75s/it]

API完单数



100%|██████████| 9/9 [04:24<00:00, 29.38s/it]


,指标,城市,对照组,实验组,P值,绝对提升,相对提升,绝对提升置信区间下界,绝对提升置信区间上界
0,冒泡数,整体,348179.714286,349758.601508,6.206661e-02,1578.887222,0.004535,-7.969698e+01,3.237471e+03
1,发单数,整体,255837.714286,257116.194394,8.195550e-02,1278.480109,0.004997,-1.621016e+02,2.719062e+03
2,完单数,整体,179456.428571,179876.804097,1.191247e-01,420.375526,0.002342,-1.083112e+02,9.490622e+02
3,应答率,整体,0.922656,0.922559,9.721775e-01,-0.000097,-0.000105,-6.234556e-01,6.232610e-01
4,成交率,整体,0.701446,0.703302,6.474743e-02,0.001855,0.002645,-4.733549e-01,4.770655e-01
5,应答后司机取消率(含改派),整体,0.079370,0.078972,3.676344e-01,-0.000398,-0.005015,-1.238392e-01,1.230432e-01
6,应答后司机取消率(不含改派),整体,0.009660,0.009559,3.034073e-01,-0.000101,-0.010447,-1.504197e-02,1.484015e-02
7,应答后乘客取消率,整体,0.156558,0.155552,1.272327e-01,-0.001006,-0.006426,-2.441484e-01,2.421365e-01
8,自营完单数,整体,46247.714286,45766.644192,4.991931e-04,-481.070093,-0.010402,-7.519251e+02,-2.102151e+02
9,宏鹄完单数,整体,80350.142857,81207.322510,3.313483e-06,857.179653,0.010668,4.959022e+02,1.218457e+03
